# Notebook 3: Embeddings from Scratch

**Goal**: Build intuition for token and positional embeddings.
Understand why embeddings work and visualize the embedding space.

After this notebook you will:
- Know how `nn.Embedding` works under the hood
- Understand positional encoding (sinusoidal and learned)
- Be able to compute cosine similarity between token vectors

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import math

from 03_embeddings.token_embedding import TokenEmbedding
from 03_embeddings.positional_encoding import SinusoidalPositionalEncoding, LearnedPositionalEmbedding

torch.manual_seed(42)
print('Imports OK')

## 1. Token embedding — the lookup table

In [ ]:
vocab_size = 100
d_model = 8   # tiny for visibility

emb = TokenEmbedding(vocab_size=vocab_size, d_model=d_model)
print(f'Embedding table shape: {emb.weight.shape}  (vocab_size x d_model)')
print(f'Total parameters: {emb.weight.numel():,}')

# Lookup token 5
token_id = torch.tensor([[5]])
vec = emb(token_id)
print(f'\nToken ID {token_id.item()} -> vector shape: {vec.shape}')
print(f'Vector: {vec[0, 0].detach().numpy().round(3)}')

In [ ]:
# Batch forward pass
# Sentence: "marathon pace was fast" -> token IDs [12, 45, 8, 31]
tokens = torch.tensor([[12, 45, 8, 31]])
embeddings = emb(tokens)
print(f'Input shape:  {tokens.shape}     [B=1, T=4]')
print(f'Output shape: {embeddings.shape}  [B=1, T=4, d_model=8]')
print('\nEmbedding matrix for this sequence:')
print(embeddings[0].detach().numpy().round(3))

## 2. Sinusoidal positional encoding — visualized

In [ ]:
d_model = 64
max_len = 100

pe_layer = SinusoidalPositionalEncoding(d_model=d_model, max_seq_len=max_len, dropout=0.0)
pe_matrix = pe_layer.pe[0].detach().numpy()   # [max_len, d_model]

plt.figure(figsize=(14, 5))
plt.imshow(pe_matrix[:50, :32].T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
plt.colorbar(label='Value')
plt.xlabel('Position (token index)')
plt.ylabel('Dimension index')
plt.title('Sinusoidal positional encoding\nEach row = one dimension, each column = one position')
plt.show()

print('Key observation: each dimension oscillates at a different frequency.')
print('High dimensions (top) = slow oscillation (captures global position)')
print('Low dimensions (bottom) = fast oscillation (captures local position)')

In [ ]:
# Show a few individual dimensions
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for i, dim in enumerate([0, 1, 8, 32]):
    axes[i].plot(pe_matrix[:50, dim])
    axes[i].set_title(f'Dimension {dim}')
    axes[i].set_xlabel('Position')
    axes[i].axhline(0, color='gray', linewidth=0.5)
plt.suptitle('PE values for selected dimensions')
plt.tight_layout()
plt.show()

## 3. Positional similarity — can the model tell positions apart?

In [ ]:
# Compute cosine similarity between all position pairs
from torch.nn.functional import cosine_similarity

pe_tensor = torch.tensor(pe_matrix[:20])   # first 20 positions
sim_matrix = torch.zeros(20, 20)
for i in range(20):
    for j in range(20):
        sim_matrix[i, j] = cosine_similarity(pe_tensor[i].unsqueeze(0), pe_tensor[j].unsqueeze(0))

plt.figure(figsize=(8, 7))
plt.imshow(sim_matrix.numpy(), cmap='Blues', vmin=0, vmax=1)
plt.colorbar(label='Cosine similarity')
plt.title('Positional encoding similarity matrix\nNearby positions are more similar')
plt.xlabel('Position')
plt.ylabel('Position')
plt.show()

## 4. Token + Positional = Full Embedding

In [ ]:
d_model = 16
vocab_size = 200

token_emb = TokenEmbedding(vocab_size=vocab_size, d_model=d_model)
pos_enc = SinusoidalPositionalEncoding(d_model=d_model, max_seq_len=64, dropout=0.0)

# Simulate a sequence: [12, 45, 8, 31, 22, 67]
tokens = torch.tensor([[12, 45, 8, 31, 22, 67]])

token_vecs = token_emb(tokens)             # Step 1: lookup
full_vecs  = pos_enc(token_vecs)           # Step 2: add positional

print(f'Token embeddings shape:   {token_vecs.shape}')
print(f'After pos encoding shape: {full_vecs.shape}')
print(f'\nThese are the vectors that enter the first transformer block.')

## Exercise

1. Change d_model from 16 to 4. Replot the PE matrix. What changes?
2. The PE formula uses 10000 as the base. Change it to 100. Does the similarity matrix change?
3. After training (Phase 6), load the model and extract `token_embedding.weight`. 
   Compute cosine similarity between 'marathon' and 'race'. Are they close?
4. `LearnedPositionalEmbedding` starts random. How long until it learns meaningful positions?